# Fase 1 — Extracción de Datos

Conexión a BigQuery y construcción del Dataset Maestro de Rentabilidad.

In [16]:
from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account
from pathlib import Path
import os
import pandas as pd
from sqlalchemy import create_engine

load_dotenv(Path.cwd().parent / ".env", override = True)

client = bigquery.Client(
    credentials=service_account.Credentials.from_service_account_file(
        os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
    ),
    project=os.getenv("BIGQUERY_PROJECT_ID")
)

print("CONEXION CORRECTA, SIN ERRORES")

CONEXION CORRECTA, SIN ERRORES


¿Qué productos y categorías están destruyendo el flujo de caja de la empresa debido a su alta edad en inventario y su baja rentabilidad real tras devoluciones?

In [ ]:
'''os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
engine = create_engine(f'bigquery://{os.getenv("BIGQUERY_PROJECT_ID")}')

query = """
WITH inventory_base AS (
    SELECT 
        ii.id AS inventory_item_id,
        ii.product_id,
        p.category,
        p.brand,
        ii.cost,
        oi.sale_price,
        oi.status,
        ii.created_at AS inv_created_at,
        ii.sold_at,
        oi.returned_at,
        TIMESTAMP_DIFF(COALESCE(ii.sold_at, CURRENT_TIMESTAMP()), ii.created_at, DAY) AS days_in_inventory
    FROM `bigquery-public-data.thelook_ecommerce.inventory_items` ii
    INNER JOIN `bigquery-public-data.thelook_ecommerce.products` p 
        ON ii.product_id = p.id
    LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` oi 
        ON ii.id = oi.inventory_item_id
),
inventory_metrics AS (
    SELECT 
        *,
        (sale_price - cost) AS unit_margin,
        AVG(days_in_inventory) OVER(PARTITION BY category) AS cat_avg_days_in_inv,
        PERCENT_RANK() OVER(PARTITION BY category ORDER BY cost) AS cost_percentile_in_cat,
        COUNT(inventory_item_id) OVER(PARTITION BY product_id) AS total_sku_stock_history,
        CASE WHEN sold_at IS NULL AND TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), inv_created_at, DAY) > 180 THEN 1 ELSE 0 END AS is_dead_stock
    FROM inventory_base
)
SELECT 
    *,
    SAFE_DIVIDE(days_in_inventory, cat_avg_days_in_inv) AS inventory_age_index
FROM inventory_metrics
"""

df_master = pd.read_sql(query, engine)
df_master.to_parquet('../data/raw_master_data.parquet', index=False) # guardamos localmente la consulta

df_master.info()
'''

# Ejecutamos solo una vez, ya que hacer la consulta cada que se corre el notebook consume tiempo y cuota de google

'os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")\nengine = create_engine(f\'bigquery://{os.getenv("BIGQUERY_PROJECT_ID")}\')\n\nquery = """\nWITH inventory_base AS (\n    SELECT \n        ii.id AS inventory_item_id,\n        ii.product_id,\n        p.category,\n        p.brand,\n        ii.cost,\n        oi.sale_price,\n        oi.status,\n        ii.created_at AS inv_created_at,\n        ii.sold_at,\n        oi.returned_at,\n        TIMESTAMP_DIFF(COALESCE(ii.sold_at, CURRENT_TIMESTAMP()), ii.created_at, DAY) AS days_in_inventory\n    FROM `bigquery-public-data.thelook_ecommerce.inventory_items` ii\n    INNER JOIN `bigquery-public-data.thelook_ecommerce.products` p \n        ON ii.product_id = p.id\n    LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` oi \n        ON ii.id = oi.inventory_item_id\n),\ninventory_metrics AS (\n    SELECT \n        *,\n        (sale_price - cost) AS unit_margin,\n        AVG(days_in_inventory) OVER(

In [21]:
df_master = df_master
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487616 entries, 0 to 487615
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   inventory_item_id        487616 non-null  int64              
 1   product_id               487616 non-null  int64              
 2   category                 487616 non-null  object             
 3   brand                    487275 non-null  object             
 4   cost                     487616 non-null  float64            
 5   sale_price               180723 non-null  float64            
 6   status                   180723 non-null  object             
 7   inv_created_at           487616 non-null  datetime64[ns, UTC]
 8   sold_at                  180723 non-null  datetime64[ns, UTC]
 9   returned_at              17953 non-null   datetime64[ns, UTC]
 10  days_in_inventory        487616 non-null  int64              
 11  unit_margin  

In [23]:
check_cols = ['inventory_item_id', 'product_id', 'cost', 'inv_created_at']
print(f"Valores nulos en columnas críticas:\n{df_master[check_cols].isnull().sum()}")

print(f"\nDistribución de estados de orden:\n{df_master['status'].value_counts(dropna=False)}")

print(f"\nConteo de Stock Muerto identificado en SQL: {df_master['is_dead_stock'].sum()}")

Valores nulos en columnas críticas:
inventory_item_id    0
product_id           0
cost                 0
inv_created_at       0
dtype: int64

Distribución de estados de orden:
status
None          306893
Shipped        54337
Complete       45387
Processing     35768
Cancelled      27278
Returned       17953
Name: count, dtype: int64

Conteo de Stock Muerto identificado en SQL: 283264
